# Repair cascade: testing Lipshitz coefficient blowup on $\Phi_n$

Consider variables
$$x_1,\ldots,x_n,y$$
on the order cone
$$0 \le x_1 \le x_2 \le \cdots \le x_n \le y$$
and the divisibility system
$$\Phi_n \;:=\; (y \mid 3x_n) \;\wedge\; \bigwedge_{k=2}^{n}\,(x_k + x_{k-1} \mid y).$$
All input coefficients lie in $\{0,1,3\}$.

Define $P_0 := 3$ and $P_{i+1} := P_i (P_i - 1)$. The accompanying derivation predicts that a sequence of quotient-aware repairs produces an accumulated equality $y = P_i \, x_{n-i}$:

| $i$ | $P_i$ |
|----:|------:|
| 0 | 3 |
| 1 | 6 |
| 2 | 30 |
| 3 | 870 |
| 4 | 756 030 |
| 5 | $\approx 5.72 \cdot 10^{11}$ |

Even though every coefficient of the input is in $\{0,1,3\}$, the cascade *would* generate exponentially large constants if it is unavoidable on every branch.

**Question for this notebook.** Does the original Lipshitz normalizer (`LinDivs.norm`) produce coefficient blowup on every SAT branch of $\Phi_n$, $n \in \{2,3,4,5\}$, or is there a compact branch that escapes the cascade?

For each SAT leaf we classify the residual divisibility system $(F, G)$:

* **empty** — every constraint was substituted out, $F = G = \emptyset$,
* **tautological** — $F = G$ rowwise, so every constraint is $f \mid f$,
* **non-trivial residual** — at least one $F_i \neq G_i$ remains; we report $\max_{c \in F \cup G} |c|$ as the residual coefficient size.

A non-trivial residual leaf with $\max |c| < P_{n-1}$ exhibits a compact escape from the cascade.

In [ ]:
import contextlib
import io
import os
import sys
import time

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.basename(os.getcwd()) != "examples":
    REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from systems.lindivs import LinDivs, OrdLinDivs
from tests.test_norm_xval import (
    _with_timeout, _NormTimeout, encode_leaf, _check,
)

print("repo root:", REPO_ROOT)

In [ ]:
def repair_cascade(n: int) -> LinDivs:
    """Build $\\Phi_n$ as a LinDivs.

    Variable indices: x_1..x_n -> 0..n-1; y -> n.
    Vector length = n + 2 (n+1 variables, 1 constant slot).

      y | 3 * x_n             (1 divisibility)
      (x_k + x_{k-1}) | y     for k = 2..n   (n - 1 divisibilities)
    Total: n divisibilities, input coefficients in {0,1,3}.
    """
    assert n >= 2
    veclen = (n + 1) + 1

    def vec(pairs, const=0):
        v = [0] * veclen
        for i, c in pairs:
            v[i] = c
        v[-1] = const
        return tuple(v)

    F, G = [], []
    F.append(vec([(n, 1)]))
    G.append(vec([(n - 1, 3)]))
    for k in range(2, n + 1):
        F.append(vec([(k - 1, 1), (k - 2, 1)]))
        G.append(vec([(n, 1)]))
    return LinDivs(tuple(F), tuple(G))


lds3 = repair_cascade(3)
print("Phi_3 has", len(lds3.F), "divisibility constraints:")
print(lds3)

In [ ]:
def lipshitz_coef_metrics(leaf):
    """Per-leaf coefficient sizes for an OrdLinDivs (Lipshitz) leaf.

    The Lipshitz normalizer substitutes equalities *into* F and G, so
    cascade blowup, if it occurs, lives in the residual coefficients of
    leaf.F and leaf.G (including the affine constants in slot -1).
    A leaf with F == G rowwise has every constraint of the form f | f,
    which is trivially satisfied and not a meaningful residual.
    """
    F, G = leaf.F, leaf.G
    coeffs = [c for row in F for c in row] + [c for row in G for c in row]
    tautological = (len(F) > 0 and tuple(F) == tuple(G))
    if not coeffs:
        return {"n_F": 0, "n_G": 0, "max_abs": 0,
                "max_bits": 0, "total_bits": 0,
                "tautological": False}
    max_abs = int(max(abs(c) for c in coeffs))
    return {
        "n_F": len(F),
        "n_G": len(G),
        "max_abs": max_abs,
        "max_bits": max_abs.bit_length(),
        "total_bits": sum(int(abs(c)).bit_length() for c in coeffs),
        "tautological": tautological,
    }


def P(i: int) -> int:
    p = 3
    for _ in range(i):
        p = p * (p - 1)
    return p


for i in range(6):
    print(f"  P_{i} = {P(i)}")

In [ ]:
def analyse(n: int, timeout_s: int = 120,
            check_sym_inc: bool = True,
            use_all_cx_inc: bool = False) -> dict:
    """Run Lipshitz .norm() on Phi_n, SAT-filter, classify each SAT leaf as
    empty / tautological / non-trivial residual, and report per-class
    coefficient-size aggregates."""
    lds = repair_cascade(n)
    t0 = time.time()
    try:
        sink = io.StringIO()
        with contextlib.redirect_stdout(sink):
            leaves = _with_timeout(timeout_s, lds.norm,
                                   check_sym_inc=check_sym_inc,
                                   use_all_cx_inc=use_all_cx_inc)
    except _NormTimeout:
        return {"n": n, "status": "TIMED_OUT",
                "timeout_s": timeout_s, "wall_s": time.time() - t0}
    except Exception as e:
        return {"n": n, "status": "ERROR",
                "error": repr(e), "wall_s": time.time() - t0}
    wall = time.time() - t0

    n_sat = n_unsat = n_unknown = 0
    sat_trivial = sat_taut = 0
    sat_nontriv = []
    for leaf in leaves:
        r = _check(encode_leaf(leaf))
        if r is True:
            n_sat += 1
            if isinstance(leaf, OrdLinDivs):
                m = lipshitz_coef_metrics(leaf)
            else:
                m = {"n_F": 0, "n_G": 0, "max_abs": 0,
                     "max_bits": 0, "total_bits": 0,
                     "tautological": False}
            if m["n_F"] == 0:
                sat_trivial += 1
            elif m["tautological"]:
                sat_taut += 1
            else:
                sat_nontriv.append((leaf, m))
        elif r is False:
            n_unsat += 1
        else:
            n_unknown += 1

    nt_min = (min(sat_nontriv, key=lambda lm: lm[1]["max_abs"])
              if sat_nontriv else None)
    nt_max = (max(sat_nontriv, key=lambda lm: lm[1]["max_abs"])
              if sat_nontriv else None)

    return {
        "n": n,
        "status": "ok",
        "n_leaves": len(leaves),
        "n_sat": n_sat, "n_unsat": n_unsat, "n_unknown": n_unknown,
        "n_sat_trivial": sat_trivial,
        "n_sat_tautological": sat_taut,
        "n_sat_nontrivial": len(sat_nontriv),
        "P_floor": P(n - 1),
        "P_cap": P(n),
        "nt_min_max_abs": nt_min[1]["max_abs"] if nt_min else None,
        "nt_max_max_abs": nt_max[1]["max_abs"] if nt_max else None,
        "nt_min_leaf": nt_min[0] if nt_min else None,
        "nt_max_leaf": nt_max[0] if nt_max else None,
        "wall_s": wall,
    }

In [ ]:
TIMEOUTS = {2: 60, 3: 60, 4: 120, 5: 240}
rows = []
for n in (2, 3, 4, 5):
    print(f"--- n = {n} (timeout {TIMEOUTS[n]}s) ---")
    r = analyse(n, timeout_s=TIMEOUTS[n])
    rows.append(r)
    if r["status"] != "ok":
        print(f"  {r['status']} after {r['wall_s']:.1f}s")
        continue
    print(
        f"  leaves={r['n_leaves']} (sat={r['n_sat']} "
        f"[empty={r['n_sat_trivial']}, taut={r['n_sat_tautological']}, "
        f"nontriv={r['n_sat_nontrivial']}], unsat={r['n_unsat']}, "
        f"unknown={r['n_unknown']})\n"
        f"  P_floor = P_{n - 1} = {r['P_floor']}, "
        f"P_cap = P_{n} = {r['P_cap']}\n"
        f"  non-trivial residual max_abs: "
        f"min={r['nt_min_max_abs']}, max={r['nt_max_max_abs']}\n"
        f"  wall={r['wall_s']:.2f}s"
    )

In [ ]:
print("n | leaves | SAT | empty | taut | nontriv | nt min |c| | nt max |c| | P_{n-1}")
print("--+--------+-----+-------+------+---------+------------+------------+--------")
compact_witnesses = []
for r in rows:
    if r["status"] != "ok":
        print(f"{r['n']} | {r['status']}")
        continue
    n_nt = r["n_sat_nontrivial"]
    nmin = r["nt_min_max_abs"]
    nmax = r["nt_max_max_abs"]
    print(
        f"{r['n']} | {r['n_leaves']:>6} | {r['n_sat']:>3} | "
        f"{r['n_sat_trivial']:>5} | {r['n_sat_tautological']:>4} | "
        f"{n_nt:>7} | {str(nmin):>10} | {str(nmax):>10} | {r['P_floor']:>6}"
    )
    if nmin is not None and nmin < r["P_floor"]:
        compact_witnesses.append(r)

print()
if compact_witnesses:
    print("=== COMPACT NON-TRIVIAL RESIDUAL WITNESSES ===")
    for r in compact_witnesses:
        print(f"\nn = {r['n']}, non-trivial residual max|c| = {r['nt_min_max_abs']}:")
        print(r["nt_min_leaf"])
    print("\nVerdict: NOT all branches blow up. Some non-trivial residual SAT"
          "\nleaves carry coefficients strictly below the cascade floor P_{n-1}.")
else:
    print("=== NO COMPACT NON-TRIVIAL ESCAPE ===")
    print("Every conclusive SAT branch either substituted to empty,"
          " reduced to tautologies, or carried >= P_{n-1}.")

## Caveats

1. **Substituted-out equalities.** The Lipshitz normalizer substitutes equalities into $F$ and $G$ rather than accumulating them in a kernel module. A SAT leaf with empty residual or with $F = G$ rowwise (tautological) is *not* a counterexample to the cascade — those branches just fully resolved their constraints. Counterexamples to the cascade are SAT leaves whose residual still carries non-tautological divisibilities but with coefficients $< P_{n-1}$.

2. **Branching policy.** This notebook uses `check_sym_inc=True, use_all_cx_inc=False` (the demo defaults from `example.py`), which short-circuits at the first symbolically-increasing order per subsystem and uses one non-increasing witness per disjunction. Rerunning with `use_all_cx_inc=True` (or `check_sym_inc=False`) explores a wider branch tree; if no compact branch appears under the broader policy, the cascade conjecture survives that wider test.

3. **z3 unknowns.** Per-leaf SAT checks run under a tight z3 timeout. Leaves whose status is `unknown` are reported separately and excluded from the SAT statistics.

4. **Wall-clock timeouts.** `LinDivs.norm` is wrapped in a SIGALRM-based wall-clock timeout (`_with_timeout` from `tests/test_norm_xval.py`). At $n = 4$ and $n = 5$ the branch tree explodes ($\approx 10\times$ growth per $n$), so the analysis times out and those rows are inconclusive under the chosen budget.

5. **Bug surfaced during this notebook.** `LinDivs.norm()` on $\Phi_n$ for $n \geq 3$ used to raise `AssertionError` inside `utils.matutils.matmul` because `utils.matutils.basis_of_ker` returned a bare `()` on a trivial kernel, losing the column-count of $M$. It has been fixed to return a shape-preserving $(\,(\,), \ldots, (\,)\,)$ with one empty row per column. A regression test pins the repaired behaviour: see `tests/test_lindivs.py::TestLinDivs::test_norm_handles_empty_kernel_basis` and the unit tests in `tests/test_matutils.py`.